# PhishGuard – Sinh bản nháp dữ liệu chat bằng Qwen3 (Giai đoạn 2) · bản code `7aad3031`

**Trước khi chạy** (khung bên phải của Kaggle → *Settings*): *Accelerator* **GPU T4 x2**, *Internet* **On**.
Không cần thêm dataset: mã nguồn và dữ liệu đã nhúng sẵn trong ô 2.

Chạy lần lượt từng ô. Ô 4 mặc định `LIMIT = 20` để **thử trước** (rải đều 5 nhóm): xem chất lượng ở ô 5,
ổn rồi đặt `LIMIT = 0` và chạy lại ô 4 (mẫu đã có được giữ, không sinh lại).

Kết quả: `/kaggle/working/drafts.jsonl` → tải về (khung *Output* bên phải), chép vào `training/chat_data/data/`.
Phiên Kaggle tối đa ~12 giờ; nên tải `drafts.jsonl` về định kỳ. **Chạy tiếp** lần sau: *Add Input* → *Upload*
file `drafts.jsonl` đã tải về, ô 2 sẽ tự nạp lại.

In [ ]:
# Ô 1 – Cấu hình
MODEL = "Qwen/Qwen3-14B-AWQ"  # model sinh nháp. Kiểm tra trên huggingface.co/Qwen: bản Instruct mới nhất có AWQ,
                              # giấy phép Apache 2.0. Hết VRAM thì đổi "Qwen/Qwen3-8B-AWQ".
TP = 2           # số GPU (T4 x2)
WORKERS = 32     # số yêu cầu gửi song song (vLLM tự gộp lại xử lý)
PORT = 8000

In [ ]:
# Ô 2 – Giải nén mã nguồn + dữ liệu (nhúng sẵn, bản 7aad3031) vào /kaggle/working/chat_data
import base64, io, os, shutil, zipfile

WORK = "/kaggle/working/chat_data"
PAYLOAD = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(WORK)  # ghi đè code cũ nếu có
for d, _, files in os.walk("/kaggle/input", followlinks=True):  # chạy tiếp: nạp drafts.jsonl đã tải lên
    if "drafts.jsonl" in files and not os.path.exists(f"{WORK}/data/drafts.jsonl"):
        shutil.copy(os.path.join(d, "drafts.jsonl"), f"{WORK}/data/drafts.jsonl")
        print("Chạy tiếp từ", os.path.join(d, "drafts.jsonl"))
print("Bản code 7aad3031:", sorted(os.listdir(WORK)), os.listdir(f"{WORK}/data"))

In [ ]:
# Ô 3 – Cài vLLM và bật server kiểu OpenAI (lần đầu mất ~5–10 phút: cài thư viện + tải model)
!pip install -q vllm
# vLLM kéo PyTorch bản CUDA mới, lệch với TorchAudio có sẵn của Kaggle → transformers nạp TorchAudio là lỗi.
# Không cần xử lý âm thanh nên gỡ đi.
!pip uninstall -y -q torchaudio
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU dùng được:", torch.cuda.is_available())
assert torch.cuda.is_available(), "PyTorch không thấy GPU (driver Kaggle cũ hơn bản CUDA của PyTorch?) — gửi dòng trên"
import subprocess, time, urllib.request

log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--port", str(PORT),
     "--tensor-parallel-size", str(TP), "--dtype", "half",          # T4 không có bfloat16
     "--max-model-len", "8192", "--gpu-memory-utilization", "0.92"],
    stdout=log, stderr=subprocess.STDOUT)
ready = False
for i in range(180):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5)
        ready = True
        print(f"Server sẵn sàng sau {i * 10} giây"); break
    except Exception:
        if server.poll() is not None:
            break  # vLLM đã dừng: in log bên dưới
        time.sleep(10)
log.flush()
if not ready:
    print("".join(open("/kaggle/working/vllm.log", encoding="utf-8", errors="replace").readlines()[-60:]))
    raise RuntimeError("vLLM không chạy được — gửi 60 dòng log ở trên")

In [ ]:
# Ô 4 – Sinh bản nháp (LIMIT = 20 để thử; ổn rồi đặt 0 và chạy lại ô này)
LIMIT = 20
ARGS = f"--base-url http://localhost:{PORT}/v1 --model {MODEL} --workers {WORKERS} --no-think --temperature 0.8"
if LIMIT:
    ARGS += f" --limit {LIMIT}"
!cd $WORK && python generate.py $ARGS
!cp $WORK/data/drafts.jsonl /kaggle/working/drafts.jsonl

In [ ]:
# Ô 5 – Thống kê + xem vài mẫu
!cd $WORK && python check.py data/drafts.jsonl
import json, random
rows = [json.loads(l) for l in open(f"{WORK}/data/drafts.jsonl", encoding="utf-8")]
for r in random.sample(rows, min(3, len(rows))):
    print("=" * 80, "\n", r["id"], r.get("flags"))
    for m in r["messages"]:
        print(f"--- {m['role']}:\n{m['content'][:1200]}")